# TE-HGBC — target-encoded HistGradientBoosting

`HistGradientBoostingClassifier` at a slow / shallow / heavily-regularized operating point,
plus **per-fold target encoding of all 13 columns — including the 7 numerics factorized to
their exact values**.

The two ingredients and what each is worth (same seed-42 folds throughout):

**Why per-value target encoding works here:** the synthetic labels were sampled from
probabilities that depend on *exact* feature values, and each distinct value repeats hundreds
of times in 690k rows — so per-value label rates estimate the generator's probability directly.
HGBC alone can't fully see this: it quantizes numerics into ≤255 histogram bins, merging
neighboring values (e.g. `sleep_duration` has more distinct values than bins). The TE features
hand it the per-value statistics without binning loss. Encoding is leak-free: fitted inside
each training fold with `TargetEncoder(cv=5)` internal cross-fitting.

Operating-point + TE recipe adapted from
[Health Risk - XGBoost + RealMLP blend](https://www.kaggle.com/code/szymonkapiski) /
[Health | Stacked HGBC/CatB/XGB/LGBM | Baseline](https://www.kaggle.com/code/kospintr/health-stacked-hgbc-catb-xgb-lgbm-baseline) /
[yekenot's RealMLP](https://www.kaggle.com/code/yekenot/ps-s6-e7-realmlp-pytorch) (TE idea).

## Setup Libraries

In [ ]:
import os, glob
import warnings
import numpy as np, pandas as pd
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import TargetEncoder

warnings.filterwarnings('ignore')
import json
import joblib
from pathlib import Path


## Load Data

In [ ]:
def find_data():
    if os.path.exists('data/train.csv'):
        return 'data'
    for h in glob.glob('/kaggle/input/**/train.csv', recursive=True):
        d = os.path.dirname(h)
        if os.path.exists(os.path.join(d, 'test.csv')):
            return d
    raise FileNotFoundError('competition data not found')
DATA = find_data()
print('data dir:', DATA)
train = pd.read_csv(f'{DATA}/train.csv')
test = pd.read_csv(f'{DATA}/test.csv')
print("Train shape:", train.shape)
print("Test shape :", test.shape)

## Preprocess Features

In [ ]:
ID = 'id'
TARGET = 'health_condition'
CLASS_NAMES = ['at-risk', 'unhealthy', 'fit']
y = train[TARGET].map({c: i for i, c in enumerate(CLASS_NAMES)}).to_numpy()
RAW = [c for c in train.columns if c not in (ID, TARGET)]
STR_CATS = ['diet_type', 'stress_level', 'sleep_quality',
            'physical_activity_level', 'smoking_alcohol', 'gender']

# model view: native categoricals + native NaN routing, no imputation
X, X_test = train[RAW].copy(), test[RAW].copy()
for c in STR_CATS:
    both = pd.concat([X[c], X_test[c]], ignore_index=True).fillna('NA').astype(str)
    categories = pd.Index(both.unique())
    X[c] = pd.Categorical(X[c].fillna('NA').astype(str), categories=categories).codes
    X_test[c] = pd.Categorical(X_test[c].fillna('NA').astype(str), categories=categories).codes

# TE view: every column as a string category — numerics keep their EXACT values
Xs = train[RAW].astype(str).fillna('na')
Xs_test = test[RAW].astype(str).fillna('na')
print("TE cardinalities:")
print(Xs.nunique().sort_values(ascending=False).to_string())

## Parameter Configuration

In [ ]:
CONFIG = dict(
    n_estimators=500,
    min_samples_leaf=2,
    max_features=0.80,
    bootstrap=True,
    max_samples=0.80,
    n_jobs=5,
    class_weight=None,
)
FOLD_SEEDS = [2027] 
FOLDS = 5  

def balanced_weights(yy):
    counts = np.bincount(yy, minlength=3)
    return (len(yy) / (3 * counts))[yy]
EXPERIMENT_ID = "te_extra_cpu_seed2027"


## Train K-Fold

In [ ]:
%%time                  
oof_preds  = np.zeros((len(X), 3))
test_preds = np.zeros((len(X_test), 3))
te_names = [f'te_{c}_{k}' for c in RAW for k in range(3)]
MODEL_DIR = Path('model_checkpoints')
MODEL_DIR.mkdir(exist_ok=True)
seed_scores = {}

for seed in FOLD_SEEDS:
    skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=seed)
    oof_s = np.zeros((len(X), 3))
    test_s = np.zeros((len(X_test), 3))
    for fold, (tr, va) in enumerate(skf.split(X, y), 1):
        enc = TargetEncoder(cv=5, smooth='auto', shuffle=True, random_state=seed)
        A_tr = pd.concat([X.iloc[tr].reset_index(drop=True),
                          pd.DataFrame(enc.fit_transform(Xs.iloc[tr], y[tr]), columns=te_names)], axis=1)
        A_va = pd.concat([X.iloc[va].reset_index(drop=True),
                          pd.DataFrame(enc.transform(Xs.iloc[va]), columns=te_names)], axis=1)
        A_te = pd.concat([X_test.reset_index(drop=True),
                          pd.DataFrame(enc.transform(Xs_test), columns=te_names)], axis=1)

        model = ExtraTreesClassifier(**CONFIG, random_state=seed)
        model.fit(A_tr, y[tr], sample_weight=balanced_weights(y[tr]))
        oof_s[va] = model.predict_proba(A_va)
        test_s += model.predict_proba(A_te) / FOLDS

    s = balanced_accuracy_score(y, oof_s.argmax(1))
    print(f'seed {seed} | 5-fold OOF: {s:.5f}')
    seed_scores[str(seed)] = float(s)
    oof_preds += oof_s / len(FOLD_SEEDS)
    test_preds += test_s / len(FOLD_SEEDS)
    seed_oof_df = pd.DataFrame({ID: train[ID]})
    seed_test_df = pd.DataFrame({ID: test[ID]})
    for i, cls in enumerate(CLASS_NAMES):
        seed_oof_df[cls] = oof_s[:, i]
        seed_test_df[cls] = test_s[:, i]
    seed_oof_df.to_csv(f'oof_seed_{seed}.csv', index=False)
    seed_test_df.to_csv(f'test_preds_seed_{seed}.csv', index=False)
    seed_sub = pd.DataFrame({ID: test[ID], TARGET: np.argmax(test_s, axis=1)})
    seed_sub[TARGET] = seed_sub[TARGET].map(dict(enumerate(CLASS_NAMES)))
    seed_sub.to_csv(f'submission_seed_{seed}.csv', index=False)

overall_oof = float(balanced_accuracy_score(y, oof_preds.argmax(1)))
summary = {
    'experiment_id': EXPERIMENT_ID,
    'uses_only_competition_data': True,
    'folds': FOLDS,
    'seeds': FOLD_SEEDS,
    'seed_scores': seed_scores,
    'overall_oof_balanced_accuracy': overall_oof,
}
Path('training_summary.json').write_text(json.dumps(summary, indent=2))
print('=' * 26)
print(f'Seed-averaged OOF Score: \033[1m{balanced_accuracy_score(y, oof_preds.argmax(1)):.5f}\033[0m')
print('=' * 26)

## Create Submission

In [ ]:
oof_df = pd.DataFrame({ID: train[ID]})
test_df = pd.DataFrame({ID: test[ID]})
for i, cls in enumerate(CLASS_NAMES):
    oof_df[cls] = oof_preds[:, i]
    test_df[cls] = test_preds[:, i]
oof_df.to_csv('oof_preds.csv', index=False)
test_df.to_csv('test_preds.csv', index=False)

sub = pd.DataFrame({ID: test[ID], TARGET: np.argmax(test_preds, axis=1)})
sub[TARGET] = sub[TARGET].map(dict(enumerate(CLASS_NAMES)))
sub.to_csv('submission.csv', index=False)
print('wrote submission.csv')
print(sub[TARGET].value_counts(normalize=True).round(4))
sub.head()